# Day 19 — Practice Session · **SOLUTIONS**
### Missing Values & Duplicates · Python for Data Science

**Prepared by Srinivasa Sai Chava**  ·  Boston University

---

> **Instructor copy.** Every question is followed by the answer and the reasoning.
> The student copy (`Day19_Practice_Questions.ipynb`) is identical minus the answer blocks.

| Part | Focus | Questions |
|---|---|---|
| A | Predict the output | 8 |
| B | Spot & fix the bug | 4 |
| C | Write the code | 5 |
| D | Challenge | 2 |

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import io, warnings
warnings.simplefilter("ignore")

CSV = (
    "emp_id,name,dept,salary,years\n"
    "1,Ravi, Sales,52000,5\n"
    "2,Sara,sales,-999,3\n"
    "3,Amit,IT,48000,\n"
    "3,amit ,IT,49000,7\n"
    "4,Neha,N/A,unknown,2\n"
    "5,Kiran,IT,61000,9\n"
)

raw = pd.read_csv(io.StringIO(CSV))
print("pandas", pd.__version__)
raw

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import io, warnings
warnings.simplefilter("ignore")

CSV = (
    "emp_id,name,dept,salary,years\n"
    "1,Ravi, Sales,52000,5\n"
    "2,Sara,sales,-999,3\n"
    "3,Amit,IT,48000,\n"
    "3,amit ,IT,49000,7\n"
    "4,Neha,N/A,unknown,2\n"
    "5,Kiran,IT,61000,9\n"
)

raw = pd.read_csv(io.StringIO(CSV))
print("pandas", pd.__version__)
raw

---
# Part A — Predict the Output  *(8 min)*

---
# Part A — Predict the Output

**Teaching note:** A1 and A2 are the disguised-missing pair. A5 (the shrinking spread) is
the one that changes how students think about filling.

### A1. How many missing values does pandas see?

In [ ]:
print(raw.isna().sum())

*Your prediction:*  

> ### ✅ Answer A1
> ```
> emp_id    0
> name      0
> dept      1
> salary    0
> years     1
> ```
> **Only two** — and both of the `salary` problems were missed.
>
> `-999` is a perfectly valid number as far as pandas is concerned, and `"unknown"` is a
> perfectly valid string. Neither is `NaN`, so `isna()` walks straight past them. The `dept`
> gap was found only because `"N/A"` happens to be on pandas' default list.
>
> **The real count of missing values here is four**, not two.

### A2. What are the dtypes, and why?

In [ ]:
print(raw.dtypes)

*Your prediction:*  

> ### ✅ Answer A2
> `salary` is **text**, not a number — because the word `"unknown"` is sitting in it, and a
> column cannot be part number and part word. `years` is `float64` because its one genuine
> blank forced it off integers.
>
> **The dtype is the clue.** A column you expect to be numeric arriving as text almost
> always means a sentinel value is hiding in it. Before trusting any column, run:
>
> ```python
> df.dtypes               # text where you expected numbers?
> df["salary"].describe() # a min of -999 is a flag
> df["salary"].value_counts()
> ```

### A3. Now how many?

In [ ]:
clean = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
print(clean.isna().sum())
print(clean["salary"].dtype)

*Your prediction:*  

> ### ✅ Answer A3
> ```
> emp_id    0
> name      0
> dept      1
> salary    2
> years     1
> dtype: int64
> float64
> ```
> **All four gaps are now visible**, and there is a bonus: `salary` has become `float64`.
> Removing the word `"unknown"` removed the reason the column was text.
>
> Declaring your sentinel values at read time is one argument, and it fixes both problems at
> once. Add to the list anything your source uses: `"-"`, `"missing"`, `"n/a"`, `9999`.

### A4. How many rows survive?

In [ ]:
clean = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
print(len(clean))
print(len(clean.dropna()))
print(len(clean.dropna(subset=["salary"])))

*Your prediction:*  

> ### ✅ Answer A4
> ```
> 6
> 3
> 4
> ```
> **A bare `dropna()` halves the data.** Six rows become three, because a gap in *any*
> column removes the whole row — Sara's missing salary, Amit's missing years, and Neha's
> missing dept and salary take out three rows between them.
>
> `dropna(subset=["salary"])` removes only the two rows actually missing a salary, keeping
> Amit even though his `years` is blank. **That is almost always what you meant.**

### A5. What happens to the mean and the spread?

In [ ]:
s = pd.Series([10., 20., np.nan, 40., 50.])
print(round(s.mean(), 2), round(s.std(), 2), s.count())
f = s.fillna(s.mean())
print(round(f.mean(), 2), round(f.std(), 2), f.count())

*Your prediction:*  

> ### ✅ Answer A5
> ```
> 30.0 18.26 4
> 30.0 15.81 5
> ```
> **The mean is unchanged — and that is the trap.** Filling with the mean cannot move the
> mean, so the summary statistic people check looks fine.
>
> **The spread shrank**, 18.26 → 15.81. The filled value sits exactly at the centre and adds
> no variation at all, so the data now looks *more consistent* than it really is. Any model
> trained on it will be overconfident, and any confidence interval will be too narrow.
>
> This is why reporting `std` before and after matters. The mean will not warn you.

### A6. And this one?

In [ ]:
s = pd.Series([10., 20., np.nan, 40., 50.])
print(round(s.fillna(0).mean(), 2))
print(s.interpolate().tolist())

*Your prediction:*  

> ### ✅ Answer A6
> ```
> 24.0
> [10.0, 20.0, 30.0, 40.0, 50.0]
> ```
> **`fillna(0)` moved the mean from 30.0 to 24.0.** Zero is a *value*, not an absence. A
> missing temperature is not 0 degrees; a missing salary is not an unpaid employee.
>
> Use `fillna(0)` only where missing genuinely means **none** — items sold, errors logged.
>
> `interpolate()` guessed 30, which here is exactly right because the series is a straight
> line. On real data it is still a guess — a well-reasoned one for ordered numeric series,
> and meaningless for unordered rows.

### A7. How many rows after each?

In [ ]:
dd = pd.DataFrame({"id": [1, 2, 2, 3, 3],
                   "name": ["A", "B", "B", "C", "C-diff"]})
print(len(dd.drop_duplicates()))
print(len(dd.drop_duplicates(subset="id")))
print(len(dd.drop_duplicates(subset="id", keep=False)))

*Your prediction:*  

> ### ✅ Answer A7
> ```
> 4
> 3
> 1
> ```
> **Why each differs:**
> - **No subset** — a row is a duplicate only if *every* column matches. Only the second `B`
>   goes, because `C` and `C-diff` differ. 5 → 4.
> - **`subset="id"`** — now id 3 counts as duplicated too, so one of the two `C` rows goes.
>   5 → 3. **You have just silently chosen which version of C to keep.**
> - **`keep=False`** — removes *all* copies of any duplicated id, keeping neither. Only id 1
>   survives. 5 → 1.
>
> `keep=False` is for **investigating**, not cleaning: use it to look at every conflicting
> copy before deciding.

### A8. How many distinct cities?

In [ ]:
fz = pd.Series(["Pune", " pune", "PUNE", "Mumbai"])
print(fz.nunique())
print(fz.str.strip().str.lower().nunique())

*Your prediction:*  

> ### ✅ Answer A8
> ```
> 4
> 2
> ```
> **There are two cities, and pandas sees four.** `drop_duplicates` and `nunique` compare
> strings **exactly**, so a leading space or a capital letter makes identical values look
> different.
>
> This is why text normalisation has to come **before** deduplication — and why
> `str.strip().str.lower()` is the standard opening move on any text column.

---
# Part B — Spot & Fix the Bug  *(7 min)*

---
# Part B — Spot & Fix the Bug

**Teaching note:** none of these four raises an error. All four produce confidently wrong
data, which is exactly the point of the session.

### B1. This should give the average salary.

In [ ]:
df = pd.read_csv(io.StringIO(CSV))
print("average salary:", df["salary"].mean())

*Error:* 

*Your fix:*

> ### ✅ Answer B1
> **Error:** raises, because `salary` is a **text** column — the word `"unknown"` is in it.
>
> And had `"unknown"` not been there, it would have been *worse*: the `-999` would have been
> averaged in as a real salary, producing a confidently wrong number with no error at all.

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])

print("dtype now:", df["salary"].dtype)
print("missing   :", df["salary"].isna().sum(), "of", len(df))
print(f"average salary: {df['salary'].mean():,.0f} "
      f"(from {df['salary'].count()} of {len(df)} employees)")

# Show what the -999 would have done if it had slipped through:
sneaky = pd.Series([52000., -999., 48000., 49000., 61000.])
print()
print("with -999 treated as real:", round(sneaky.mean(), 0))
print("without it               :", round(sneaky[sneaky > 0].mean(), 0))

### B2. This should fill missing years with a sensible value.

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
df["years"] = df["years"].fillna(0)
print(df[["name", "years"]])
print("mean years:", df["years"].mean())

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B2
> **Symptom:** Amit now shows **0 years of service** — he looks like a brand-new hire — and
> the mean years drops.
> **Cause:** `fillna(0)` on a measurement. Zero is a *value*, and here it is a factually
> wrong one: we do not know Amit's tenure, and 0 is the one number we can be fairly sure is
> incorrect, since he appears in the staff list.

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])

before = df["years"].agg(["mean", "std", "count"])
df["years"] = df["years"].fillna(df["years"].median())
after = df["years"].agg(["mean", "std", "count"])

print(pd.concat([before, after], axis=1, keys=["before", "after"]).round(2))
print()
print(df[["name", "years"]])

# The median is the safer default: it is an actual plausible tenure, and it
# is not dragged around by one very long-serving employee.
#
# And note we PRINTED the cost. The count rose from 5 to 6, and the spread
# changed - anyone reading this knows exactly what happened.

### B3. This should carry the last known price forward.

In [ ]:
prices = pd.DataFrame({
    "stock": ["ZZZ", "AAA", "AAA", "ZZZ"],
    "price": [100.0, 50.0, np.nan, np.nan],
})
prices["price"] = prices["price"].ffill()
print(prices)

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B3
> **Symptom:** the last row — stock `ZZZ` — is given **50.0**, which is `AAA`'s price.
> **Cause:** `ffill` carries the previous **row's** value forward regardless of what that row
> is about. The table is not sorted by stock, so it filled across two different companies.
>
> No error, no warning, and a plausible-looking number.

In [ ]:
prices = pd.DataFrame({
    "stock": ["ZZZ", "AAA", "AAA", "ZZZ"],
    "price": [100.0, 50.0, np.nan, np.nan],
})

# Fill WITHIN each stock - Day 17's groupby doing the work
prices["price"] = prices.groupby("stock")["price"].ffill()
print(prices)

# ZZZ's gap is now filled with 100.0 (its own previous price)
# AAA's gap is filled with 50.0 (its own previous price)
#
# THE RULE: ffill assumes a meaningful row ORDER. If rows from different
# groups are interleaved, fill within the group or sort first.

### B4. This should remove duplicate employees.

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
df = df.drop_duplicates()
print(len(df), "rows")
print(df[["emp_id", "name"]])

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B4
> **Symptom:** nothing is removed — still 6 rows, with `emp_id` 3 appearing twice.
> **Cause:** two problems at once. The two rows for employee 3 are not *exactly* identical
> (`"Amit"` vs `"amit "`, and different salaries), and a bare `drop_duplicates()` needs every
> column to match.

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
start = len(df)

# 1. normalise the text FIRST
for col in ["name", "dept"]:
    df[col] = df[col].str.strip().str.lower()

# 2. LOOK at the conflict before resolving it
conflicts = df[df.duplicated(subset=["emp_id"], keep=False)]
print("conflicting rows:")
print(conflicts)

# 3. then decide which copy to keep
df = df.drop_duplicates(subset=["emp_id"], keep="last")
print()
print(f"{start} rows in, {len(df)} out")

# Two salaries for the same employee is a DATA PROBLEM, not something to
# resolve silently. keep="last" assumes the later row is the correction -
# state that assumption out loud.

---
# Part C — Write the Code  *(12 min)*

---
# Part C — Write the Code

**Teaching note:** C5 is the one to demonstrate — the before/after table is the habit this
whole session exists to build.

### C1. Find everything that is missing
Read the CSV properly and print, for each column: the number missing, the proportion
missing, and how many rows have at least one gap.

In [ ]:
# your code here

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])

report = pd.DataFrame({
    "missing":    df.isna().sum(),
    "proportion": df.isna().mean().round(2),
})
print(report)
print()
print("rows with at least one gap:", df.isna().any(axis=1).sum(), "of", len(df))

#         missing  proportion
# emp_id        0        0.00
# name          0        0.00
# dept          1        0.17
# salary        2        0.33
# years         1        0.17
#
# rows with at least one gap: 3 of 6
#
# That last number is the one to look at before reaching for dropna():
# it tells you exactly how much a bare dropna() would cost you.

### C2. Drop carefully
Using the cleaned DataFrame, compare three strategies and print the surviving row count for
each: drop any gap, drop only rows missing a salary, drop only completely empty rows.

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])

# your code here

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])

print("original            :", len(df))
print("dropna()            :", len(df.dropna()))
print("dropna(subset=sal)  :", len(df.dropna(subset=["salary"])))
print("dropna(how='all')   :", len(df.dropna(how="all")))

# original            : 6
# dropna()            : 3
# dropna(subset=sal)  : 4
# dropna(how='all')   : 6
#
# The bare dropna() loses HALF the data to answer a question about salary.
# dropna(subset=["salary"]) removes only what is genuinely unusable.
# how="all" removes nothing here, which is why it is a safe first step.

### C3. Fill three ways and compare
Take the `years` column. Fill it with 0, with the mean, and with the median. Print the mean,
std and count for each, in one table.

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
years = df["years"]

# your code here

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
years = df["years"]

rows = []
for label, filled in [
    ("leave it",       years),
    ("fillna(0)",      years.fillna(0)),
    ("fillna(mean)",   years.fillna(years.mean())),
    ("fillna(median)", years.fillna(years.median())),
]:
    rows.append({"choice": label,
                 "mean":  round(filled.mean(), 2),
                 "std":   round(filled.std(), 2),
                 "count": filled.count()})

print(pd.DataFrame(rows).set_index("choice"))

#                 mean   std  count
# choice
# leave it        5.20  2.86      5
# fillna(0)       4.33  3.33      6
# fillna(mean)    5.20  2.56      6
# fillna(median)  5.17  2.56      6
#
# READ IT:
#   fillna(0)      moved the mean DOWN (5.20 -> 4.33) and the spread UP
#                  (2.86 -> 3.33) - it invented an implausible outlier.
#   fillna(mean)   held the mean EXACTLY and shrank the spread to 2.56.
#   fillna(median) the median here is 5.0, close to the mean, so it behaves
#                  almost identically - but it would not have, had one
#                  employee served 40 years.
#
# None of these is "correct". The point is that you can now say what each cost.

### C4. Clean the text, then deduplicate
Normalise `name` and `dept`, then remove duplicate employees keeping the **last** record.
Print how many rows you removed and which `emp_id` was affected.

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])

# your code here

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
start = len(df)

# 1. normalise FIRST
for col in ["name", "dept"]:
    df[col] = df[col].str.strip().str.lower()

# 2. identify before removing
dup_ids = df.loc[df.duplicated(subset=["emp_id"], keep=False), "emp_id"].unique()
print("duplicated emp_ids:", dup_ids.tolist())
print(df[df["emp_id"].isin(dup_ids)][["emp_id", "name", "salary"]])

# 3. remove
df = df.drop_duplicates(subset=["emp_id"], keep="last")
print()
print(f"removed {start - len(df)} row(s): {start} -> {len(df)}")

# duplicated emp_ids: [3]
# removed 1 row(s): 6 -> 5
#
# Note the dept values are now 'sales' and 'it' consistently - before
# normalising, " Sales" and "sales" were two different departments.

### C5. A pipeline with a receipt
Put it all together: read with `na_values`, normalise text, deduplicate, fill `salary` with
the median, and print a before/after summary plus the row count.

In [ ]:
# your code here

In [ ]:
df = pd.read_csv(io.StringIO(CSV), na_values=["-999", "N/A", "unknown", ""])
start = len(df)

print("=== BEFORE ===")
print("rows:", start)
print(df.isna().sum().to_dict())

# normalise text before deduplicating
for col in ["name", "dept"]:
    df[col] = df[col].str.strip().str.lower()

df = df.drop_duplicates(subset=["emp_id"], keep="last")
after_dedupe = len(df)

# fill, and measure
before = df["salary"].agg(["mean", "std", "count"])
df["salary"] = df["salary"].fillna(df["salary"].median())
after = df["salary"].agg(["mean", "std", "count"])

print()
print("=== SALARY: effect of filling ===")
print(pd.concat([before, after], axis=1, keys=["before", "after"]).round(2))

print()
print("=== RECEIPT ===")
print(f"{start} rows in")
print(f"  -{start - after_dedupe} duplicate employee record(s)")
print(f"  {before['count']:.0f} salaries present, {after['count'] - before['count']:.0f} filled with the median")
print(f"{len(df)} rows out")

# Anyone reading that output can reconstruct exactly what was done.
# That is the standard - not a clean dataset, but a DOCUMENTED one.

---
# Part D — Challenge  *(3 min, or take home)*

---
# Part D — Challenge

**Teaching note:** D1 is the most important idea of the day — missingness that is not random.
If students take one thing away, make it this.

### D1. When dropping biases the answer
In the data below, compute the average salary two ways: dropping the missing rows, and after
investigating **who** is missing. Explain what dropping did to the answer.

In [ ]:
staff = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Dev", "Meena", "Arun"],
    "level":  ["junior", "exec", "junior", "exec", "junior", "exec", "junior", "junior"],
    "salary": [42000, np.nan, 38000, np.nan, 45000, np.nan, 41000, 39000],
})

# your code here

In [ ]:
staff = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Dev", "Meena", "Arun"],
    "level":  ["junior", "exec", "junior", "exec", "junior", "exec", "junior", "junior"],
    "salary": [42000, np.nan, 38000, np.nan, 45000, np.nan, 41000, 39000],
})

# ---- the naive answer
print(f"average after dropping: {staff['salary'].mean():,.0f} "
      f"(from {staff['salary'].count()} of {len(staff)})")

# ---- WHO is missing?
print()
print("levels WITH a salary:")
print(staff[staff["salary"].notna()]["level"].value_counts())
print()
print("levels MISSING a salary:")
print(staff[staff["salary"].isna()]["level"].value_counts())

# average after dropping: 41,000 (from 5 of 8)
#
# levels WITH a salary:    junior 5
# levels MISSING a salary: exec   3
#
# EVERY missing salary is an executive. The gaps are not random at all.
#
# So "the average salary is 41,000" is not an average of this company's
# staff - it is an average of its JUNIOR staff, reported as though it
# described everyone. Dropping did not just lose data, it removed the
# entire top of the distribution.
print()
print("What you should actually report:")
print(f"  junior average: {staff[staff['level'] == 'junior']['salary'].mean():,.0f} "
      f"(from {staff[staff['level'] == 'junior']['salary'].count()} of 5)")
print(f"  exec average  : no data (0 of 3 reported)")
print()
print("Filling the execs with the overall mean would be WORSE - it would")
print("assert that executives earn the same as juniors, which we know is false.")

### D2. Order matters
Using the data below, build the same pipeline two ways — deduplicate then normalise, and
normalise then deduplicate — and show that they give different results.

In [ ]:
orders = pd.DataFrame({
    "city": ["Pune", " Pune", "PUNE", "Delhi", "delhi"],
    "amt":  [100, 100, 100, 200, 200],
})

# your code here

In [ ]:
orders = pd.DataFrame({
    "city": ["Pune", " Pune", "PUNE", "Delhi", "delhi"],
    "amt":  [100, 100, 100, 200, 200],
})
print("starting rows:", len(orders))

# ---- WRONG ORDER: deduplicate first
a = orders.drop_duplicates()
a["city"] = a["city"].str.strip().str.lower()
print()
print("dedupe, then normalise:", len(a), "rows")
print(a)
print("total amount:", a["amt"].sum())

# ---- RIGHT ORDER: normalise first
b = orders.copy()
b["city"] = b["city"].str.strip().str.lower()
b = b.drop_duplicates()
print()
print("normalise, then dedupe:", len(b), "rows")
print(b)
print("total amount:", b["amt"].sum())

# dedupe, then normalise: 5 rows, total 700
# normalise, then dedupe: 2 rows, total 300
#
# WHY: at the moment of deduplication, "Pune", " Pune" and "PUNE" are three
# genuinely different strings, so drop_duplicates keeps all three. Cleaning
# them afterwards leaves you with three identical rows it is now too late
# to remove.
#
# A 700 versus 300 difference from two steps in a different order, with no
# error either way.
#
# THE RULE: normalise text BEFORE deduplicating. Always.

---
## Done? Self-check

- [ ] I know `isna()` misses `-999`, `"N/A"` and `"unknown"`
- [ ] I use `na_values=` at read time
- [ ] I check the **proportion** missing, not just the count
- [ ] I ask **why** values are missing before deciding
- [ ] I know `fillna(mean)` shrinks the spread
- [ ] I normalise text **before** deduplicating
- [ ] I printed what each cleaning step changed

### Homework
1. Take a messy CSV and write a cleaning pipeline that prints a receipt.
2. Fill one column three different ways and compare the mean and std each time.
3. Find a dataset where dropping the gaps biases the result, and say how.

### Next class — Topic 1.19: Outlier detection & treatment
The 1.5 × IQR rule, z-scores, and deciding whether a strange value is an error or the
finding.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*

---
## Wrap-up — running the last 5 minutes

Three cold-call questions:

1. *"Does `isna()` find `-999`?"* → No. Declare it in `na_values`.
2. *"What does `fillna(mean)` do to the standard deviation?"* → shrinks it.
3. *"Normalise or deduplicate first?"* → normalise, always.

**Common misconceptions to watch for today**

| Misconception | Correction |
|---|---|
| "`isna()` finds all missing values" | It misses sentinels like `-999` and `"unknown"` |
| "Dropping rows is the safe option" | It can remove an entire subgroup |
| "Filling with the mean is harmless" | The mean holds, the spread shrinks |
| "`fillna(0)` is a reasonable default" | Zero is a value; it changes the mean |
| "`ffill` just fills gaps" | It assumes the row order is meaningful |
| "`drop_duplicates()` catches duplicates" | Only exact ones — `" Pune"` ≠ `"Pune"` |
| "Cleaning steps can go in any order" | Dedupe-then-clean leaves duplicates behind |

**D1 is the one to spend time on.** Every missing salary belongs to an executive, so the
"average salary" after dropping describes only the junior staff — while being reported as
though it described everyone. Dropping did not lose a random sample; it removed the top of
the distribution. That single example does more than any amount of warning about
missingness.

**D2 gives 700 versus 300** from the same two steps in a different order, with no error
either way. It is the cleanest possible argument for having a fixed pipeline order.

**The habit to send them away with:** every cleaning step gets one line saying what it
changed. Rows in, rows out, and what happened to the mean and spread. A dataset nobody can
reconstruct is a dataset nobody can trust.

**Homework given:** a pipeline with a receipt; three fills compared; a biased-missingness
example.

**Next session:** Topic 1.19 — outliers. The same judgement problem in a new form: a value
that looks wrong may be a typo, or may be the most important observation in the dataset.